# KOA Classifier: Custom CNN Pembanding

*Skenario*: Balanced to 1000 image

## 1 — Mount, Install & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install imbalanced-learn tqdm -q

import os, cv2, time, random, warnings, shutil
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

from imblearn.under_sampling import RandomUnderSampler
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, AveragePooling2D,
    GlobalAveragePooling2D, GlobalMaxPooling2D, Concatenate,
    Dropout, BatchNormalization, Multiply, Reshape, Lambda,
    Flatten, Activation
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.utils import Sequence, to_categorical
import tensorflow.keras.backend as K

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, classification_report,
    accuracy_score, confusion_matrix, roc_curve, auc, f1_score)
from itertools import cycle

from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision aktif:", mixed_precision.global_policy())

gpus = tf.config.list_physical_devices('GPU')
print("GPU tersedia:", gpus)

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU aktif dan siap digunakan.")
else:
    print("GPU tidak terdeteksi. Cek kembali runtime Colab.")

SEED        = 100
GRADE_NAMES = ['Grade 0 (Normal)', 'Grade 1 (Meragukan)',
               'Grade 2 (Ringan)', 'Grade 3 (Sedang)', 'Grade 4 (Berat)']
GRADE_SHORT = ['Grade 0','Grade 1','Grade 2','Grade 3','Grade 4']
N_FEATURES  = 200
NUM_CLASSES = 5
BATCH_SIZE  = 64

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED']       = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

ARCHITECTURE_MODE = "reference_cnn"

VALID_ARCHITECTURES = ["vgg19_attention", "reference_cnn"]
if ARCHITECTURE_MODE not in VALID_ARCHITECTURES:
    raise ValueError(f"ARCHITECTURE_MODE harus salah satu dari: {VALID_ARCHITECTURES}")

ARCH_TAG = ARCHITECTURE_MODE
print(f"Mode arsitektur aktif: {ARCH_TAG}")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/archive"
DATASET_DIR       = "/content/archive"
SAVE_MODEL        = "/content/drive/MyDrive/skripsi/model_final"
SAVE_PRE          = "/content/preprocessing_final"

if not os.path.exists(DATASET_DIR):
    print(f"Menyalin dataset dari {DRIVE_DATASET_DIR} ke {DATASET_DIR}...")
    if not os.path.exists(DRIVE_DATASET_DIR):
        raise FileNotFoundError(f"DRIVE_DATASET_DIR tidak ditemukan: {DRIVE_DATASET_DIR}")
    shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
else:
    print(f"Dataset lokal sudah tersedia: {DATASET_DIR}")

for d in [SAVE_MODEL, SAVE_PRE+"/train", SAVE_PRE+"/val", SAVE_PRE+"/test"]:
    os.makedirs(d, exist_ok=True)

BEST_MODEL_PATH = SAVE_MODEL + f"/{ARCH_TAG}_best.weights.h5"

from IPython.display import display

REPORT_DIR = SAVE_MODEL + "/laporan_proses"
os.makedirs(REPORT_DIR, exist_ok=True)

def save_and_show(df, filename, n=10, title=None):
    path = os.path.join(REPORT_DIR, filename)
    df.to_csv(path, index=False)

    if title:
        print("\n" + title)

    display(df.head(n))
    print(f"Disimpan: {path}")
    return path

MONITOR_VERBOSE = 3
N_JOBS_TUNING   = -1
CV_FOLDS_TUNING = 5

SVM_TUNING_N_ITER = 24
RF_TUNING_N_ITER  = 32

FORCE_RERUN_GA = False

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def log_step(message):
    print(f"[{now_str()}] {message}", flush=True)

def fmt_minutes(seconds):
    return f"{seconds/60:.2f} menit"

log_step("Monitoring global aktif. Output progress akan muncul saat GA dan tuning model berjalan.")
print(f"Folder laporan proses: {REPORT_DIR}")
print(f"Setup selesai | SEED={SEED}")
print(f"Dataset digunakan dari: {DATASET_DIR}")

## 2 — Preprocessing

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest'
)

def apply_clahe(img_bgr, clip_limit=3.0, tile_grid=(8,8)):
    try:
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
        l_enh = clahe.apply(l)
        result = cv2.cvtColor(cv2.merge((l_enh, a, b)), cv2.COLOR_LAB2RGB)
    except Exception:
        result = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return result

def preprocess_image(image_path, target_size=(224,224),
                     augment=False, save_dir=None, filename_prefix=""):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Gambar tidak bisa dibaca oleh cv2.imread: {image_path}")

    img = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)

    if augment:
        img = datagen.random_transform(img)

    final = apply_clahe(img)

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        tag = "AUG_" if augment else "ORI_"
        cv2.imwrite(
            os.path.join(save_dir,
                         f"{filename_prefix}{tag}CLAHE_{os.path.basename(image_path)}"),
            cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
        )

    return preprocess_input(final.astype(np.float32))

def load_paths_and_labels(base_path, split_name='train'):
    X, y = [], []
    folder = os.path.join(base_path, split_name) if split_name else base_path
    if not os.path.exists(folder):
        print(f"Tidak ditemukan: {folder}")
        return np.array([]), np.array([])
    for lbl, name in enumerate(["0","1","2","3","4"]):
        d = os.path.join(folder, name)
        if not os.path.exists(d):
            continue
        for f in os.listdir(d):
            if f.lower().endswith(('.png','.jpg','.jpeg')):
                X.append(os.path.join(d, f))
                y.append(lbl)
    return np.array(X), np.array(y)

def get_balanced_paths(X_paths, y_labels, target_count=1000):
    if len(X_paths) == 0:
        return np.array([]), np.array([]), np.array([])

    unique, counts = np.unique(y_labels, return_counts=True)
    rus_strat = {c: target_count for c, n in zip(unique, counts) if n > target_count}

    if rus_strat:
        rus = RandomUnderSampler(sampling_strategy=rus_strat, random_state=SEED)
        Xr, yr = rus.fit_resample(X_paths.reshape(-1, 1), y_labels)
        Xr = Xr.flatten()
    else:
        Xr, yr = X_paths, y_labels

    Xf, yf, af = [], [], []
    for c in sorted(np.unique(yr)):
        idx = np.where(yr == c)[0]
        Xc, yc = Xr[idx], yr[idx]
        n = len(idx)

        Xf.extend(Xc)
        yf.extend(yc)
        af.extend([False] * n)

        if n < target_count:
            need = target_count - n
            add_idx = np.random.choice(idx, size=need, replace=True)
            Xf.extend(Xr[add_idx])
            yf.extend(yr[add_idx])
            af.extend([True] * need)

    return np.array(Xf), np.array(yf), np.array(af, dtype=bool)

## 3 — Visualisasi CLAHE per Grade

In [ ]:
def visualize_clahe_per_grade(dataset_dir, n_samples=2):
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    titles = ['Asli', 'CLAHE', 'Histogram']
    fig.suptitle('Perbandingan Citra Asli vs CLAHE per Grade KOA',
                  fontsize=14, fontweight='bold')

    for grade in range(5):
        folder = os.path.join(dataset_dir, 'train', str(grade))
        if not os.path.exists(folder): continue
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.png'))]
        if not files: continue
        np.random.seed(SEED+grade)
        path = os.path.join(folder, np.random.choice(files))

        img_bgr = cv2.imread(path)
        img_bgr = cv2.resize(img_bgr, (224,224), interpolation=cv2.INTER_CUBIC)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_enh = apply_clahe(img_bgr)

        axes[0, grade].imshow(img_rgb)
        axes[0, grade].set_title(f'Grade {grade} — Asli', fontsize=10, fontweight='bold')
        axes[0, grade].axis('off')

        axes[1, grade].imshow(img_enh)
        axes[1, grade].set_title(f'Grade {grade} — CLAHE', fontsize=10,
                                   fontweight='bold', color='#0066CC')
        axes[1, grade].axis('off')

        gray_ori = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        gray_enh = cv2.cvtColor(cv2.cvtColor(img_enh, cv2.COLOR_RGB2BGR),
                                 cv2.COLOR_BGR2GRAY)
        axes[2, grade].hist(gray_ori.flatten(), bins=64, color='salmon',
                             alpha=0.6, density=True, label='Asli')
        axes[2, grade].hist(gray_enh.flatten(), bins=64, color='steelblue',
                             alpha=0.6, density=True, label='CLAHE')
        delta = gray_enh.std() - gray_ori.std()
        axes[2, grade].set_title(f'Δσ = {delta:+.1f}', fontsize=9)
        if grade==0: axes[2, grade].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(SAVE_MODEL+'/visualisasi_clahe.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("Disimpan: visualisasi_clahe.png")

visualize_clahe_per_grade(DATASET_DIR)

## 4 — DataGenerator & Load Data

In [ ]:
class DataGenerator(Sequence):
    def __init__(self, image_paths, labels, augment_flags=None,
                 batch_size=64, target_size=(224,224),
                 shuffle=False, save_dir=None):
        self.image_paths   = np.array(image_paths)
        self.labels        = np.array(labels)
        self.augment_flags = (np.array(augment_flags) if augment_flags is not None
                              else np.zeros(len(image_paths), dtype=bool))
        self.batch_size=batch_size; self.target_size=target_size
        self.shuffle=shuffle; self.save_dir=save_dir
        self.indices=np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths)/self.batch_size))

    def __getitem__(self, idx):
        start=idx*self.batch_size
        end=min((idx+1)*self.batch_size, len(self.image_paths))
        bidx=self.indices[start:end]
        X=np.empty((len(bidx),*self.target_size,3), dtype=np.float32)
        y=np.empty(len(bidx), dtype=int)
        for i, ix in enumerate(bidx):
            X[i]=preprocess_image(self.image_paths[ix],
                                   target_size=self.target_size,
                                   augment=self.augment_flags[ix],
                                   save_dir=self.save_dir,
                                   filename_prefix=f"Grade{self.labels[ix]}_Idx{ix}_")
            y[i]=self.labels[ix]
        return X, to_categorical(y, num_classes=5)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

print("=== LOAD DATA ===")
X_train_raw, y_train_raw    = load_paths_and_labels(DATASET_DIR, 'train')
X_val_paths, y_val_labels   = load_paths_and_labels(DATASET_DIR, 'val')
X_test_paths, y_test_labels = load_paths_and_labels(DATASET_DIR, 'test')

print(f"Train asli: {len(X_train_raw):5d} | Val: {len(X_val_paths):4d} | Test: {len(X_test_paths):4d}")
for lbl in range(5):
    n_tr = np.sum(y_train_raw==lbl)
    print(f"  Grade {lbl}: {n_tr:5d} train | {np.sum(y_val_labels==lbl):4d} val | {np.sum(y_test_labels==lbl):4d} test")

print("\n=== BALANCING TRAIN (1000/kelas) ===")
X_train_bal, y_train_bal, aug_train = get_balanced_paths(
    X_train_raw, y_train_raw, target_count=1000)
print(f"Train balanced: {len(X_train_bal)} (1000 × 5 kelas)")

train_gen = DataGenerator(X_train_bal, y_train_bal,
                          augment_flags=aug_train, batch_size=64, shuffle=True)
val_gen   = DataGenerator(X_val_paths,  y_val_labels,  batch_size=64)
test_gen  = DataGenerator(X_test_paths, y_test_labels, batch_size=64)
print(f"Batch — Train:{len(train_gen)} | Val:{len(val_gen)} | Test:{len(test_gen)}")

rows_dist = []

for g in range(5):
    rows_dist.append({
        "Grade": g,
        "Nama Grade": GRADE_SHORT[g],
        "Train Asli": int(np.sum(y_train_raw == g)),
        "Train Balanced": int(np.sum(y_train_bal == g)),
        "Validasi": int(np.sum(y_val_labels == g)),
        "Test": int(np.sum(y_test_labels == g))
    })

df_dist = pd.DataFrame(rows_dist)
save_and_show(
    df_dist,
    "01_distribusi_dataset.csv",
    n=10,
    title="LAPORAN 01: Distribusi Dataset per Grade"
)

df_train_balanced = pd.DataFrame({
    "Path": X_train_bal,
    "Nama File": [os.path.basename(p) for p in X_train_bal],
    "Grade": y_train_bal,
    "Nama Grade": [GRADE_SHORT[g] for g in y_train_bal],
    "Augmentasi": aug_train.astype(bool)
})

save_and_show(
    df_train_balanced,
    "02_daftar_data_train_balanced.csv",
    n=10,
    title="LAPORAN 02: Daftar Data Train Setelah Balancing"
)

## 5 - Arsitektur CNN

In [ ]:
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    avg_pool = GlobalAveragePooling2D()(input_feature)
    max_pool = GlobalMaxPooling2D()(input_feature)

    dense1 = Dense(channel // ratio, activation='relu',
                   kernel_initializer='he_normal', use_bias=True)
    dense2 = Dense(channel, kernel_initializer='he_normal', use_bias=True)

    avg_out = dense2(dense1(avg_pool))
    max_out = dense2(dense1(max_pool))

    cbam_feature = avg_out + max_out
    cbam_feature = Activation('sigmoid')(cbam_feature)
    cbam_feature = Reshape((1, 1, channel))(cbam_feature)

    return Multiply()([input_feature, cbam_feature])

def build_vgg19_attention(n_classes=5, input_shape=(224,224,3), bottleneck_dim=200):
    base = VGG19(weights='imagenet', include_top=False,
                 input_shape=input_shape)
    base.trainable = False

    conv_out = base.output
    x_att = channel_attention(conv_out, ratio=8)

    avg_att = GlobalAveragePooling2D()(x_att)
    max_att = GlobalMaxPooling2D()(x_att)
    x = Concatenate()([avg_att, max_att])

    x = Dense(512, activation='relu', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(256, activation='relu', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(bottleneck_dim, activation='linear', name='bottleneck_features')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    output = Dense(n_classes, activation='softmax', dtype='float32', name='predictions')(x)
    model = Model(inputs=base.input, outputs=output, name='VGG19_ChannelAttention_200F')

    return model, base

def build_reference_like_cnn(n_classes=5, input_shape=(224,224,3), bottleneck_dim=200):
    inputs = Input(shape=input_shape, name='input_image')

    x = Conv2D(32, (3,3), padding='same', activation='relu',
               kernel_initializer='he_normal', name='ref_block1_conv1')(inputs)
    x = BatchNormalization(name='ref_block1_bn1')(x)
    x = Conv2D(32, (3,3), padding='same', activation='relu',
               kernel_initializer='he_normal', name='ref_block1_conv2')(x)
    x = BatchNormalization(name='ref_block1_bn2')(x)
    x = MaxPooling2D(pool_size=(2,2), name='ref_block1_maxpool')(x)
    x = Dropout(0.30, name='ref_block1_dropout')(x)

    x = Conv2D(64, (3,3), padding='same', activation='relu',
               kernel_initializer='he_normal', name='ref_block2_conv1')(x)
    x = BatchNormalization(name='ref_block2_bn1')(x)
    x = Conv2D(64, (3,3), padding='same', activation='relu',
               kernel_initializer='he_normal', name='ref_block2_conv2')(x)
    x = BatchNormalization(name='ref_block2_bn2')(x)
    x = AveragePooling2D(pool_size=(2,2), name='ref_block2_avgpool')(x)
    x = Dropout(0.30, name='ref_block2_dropout')(x)

    x = Conv2D(128, (3,3), padding='same', activation='relu',
               kernel_initializer='he_normal', name='ref_block3_conv1')(x)
    x = BatchNormalization(name='ref_block3_bn1')(x)
    x = AveragePooling2D(pool_size=(2,2), name='ref_block3_avgpool')(x)
    x = Dropout(0.30, name='ref_block3_dropout')(x)

    x = Flatten(name='flatten_features')(x)

    x = Dense(512, activation='relu', kernel_initializer='he_normal', name='ref_dense_512')(x)
    x = BatchNormalization(name='ref_dense_512_bn')(x)
    x = Dropout(0.50, name='ref_dense_512_dropout')(x)

    x = Dense(bottleneck_dim, activation='linear', name='bottleneck_features')(x)
    x = BatchNormalization(name='ref_bottleneck_bn')(x)
    x = Dropout(0.30, name='ref_bottleneck_dropout')(x)

    output = Dense(n_classes, activation='softmax', dtype='float32', name='predictions')(x)
    model = Model(inputs=inputs, outputs=output, name='ReferenceLike_CustomCNN_200F')

    return model, None

if ARCHITECTURE_MODE == 'vgg19_attention':
    model, base_model = build_vgg19_attention(
        n_classes=NUM_CLASSES,
        input_shape=(224,224,3),
        bottleneck_dim=N_FEATURES
    )
    TRAINING_STRATEGY = 'three_phase_finetuning'
elif ARCHITECTURE_MODE == 'reference_cnn':
    model, base_model = build_reference_like_cnn(
        n_classes=NUM_CLASSES,
        input_shape=(224,224,3),
        bottleneck_dim=N_FEATURES
    )
    TRAINING_STRATEGY = 'single_phase_custom_cnn'

print(f"Arsitektur berhasil dibangun: {ARCHITECTURE_MODE}")
print(f"Strategi training: {TRAINING_STRATEGY}")
print(f"Checkpoint terbaik: {BEST_MODEL_PATH}")
print()

model.summary()

trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
total = sum(tf.size(w).numpy() for w in model.weights)
print(f"Parameter trainable : {trainable:,}")
print(f"Parameter total     : {total:,}")

## 6 — Training Phase

In [ ]:
metrik = [
    tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
    tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=NUM_CLASSES),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]

loss_fn = CategoricalCrossentropy(label_smoothing=0.05)

ckpt = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

csv_logger = CSVLogger(
    os.path.join(REPORT_DIR, f"training_log_{ARCH_TAG}.csv"),
    append=False
)

history_p1 = None
history_p2 = None
history_p3 = None

if ARCHITECTURE_MODE == 'vgg19_attention':
    print("=" * 55)
    print("  FASE 1: WARM-UP (10 epoch)")
    print("  Backbone frozen, latih kepala + attention")
    print("=" * 55)

    base_model.trainable = False
    model.compile(optimizer=Adam(learning_rate=1e-3),
                  loss=loss_fn, metrics=metrik)

    history_p1 = model.fit(
        train_gen, validation_data=val_gen, epochs=10,
        callbacks=[
            EarlyStopping(monitor='val_auc', mode='max', patience=5,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_auc', mode='max',
                              factor=0.5, patience=2, min_lr=1e-7, verbose=1),
            ckpt,
            csv_logger
        ]
    )

    print("\n" + "=" * 55)
    print("  FASE 2: FINE-TUNING (Block 5, max 20 epoch)")
    print("  block5_conv1 sampai block5_conv4 dibuka")
    print("=" * 55)

    base_model.trainable = True
    set_trainable = False
    for layer in base_model.layers:
        if layer.name == 'block5_conv1':
            set_trainable = True
        layer.trainable = set_trainable

    opened = [l.name for l in base_model.layers if l.trainable]
    print(f"Layer dibuka: {opened}")

    model.compile(optimizer=SGD(learning_rate=1e-5, momentum=0.9),
                  loss=loss_fn, metrics=metrik)

    history_p2 = model.fit(
        train_gen, validation_data=val_gen, epochs=20,
        callbacks=[
            EarlyStopping(monitor='val_auc', mode='max', patience=5,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_auc', mode='max',
                              factor=0.5, patience=2, min_lr=1e-7, verbose=1),
            ckpt
        ]
    )

    print("\n" + "=" * 55)
    print("  FASE 3: FINE-TUNING (Block 4+5, max 15 epoch)")
    print("  Monitor: val_loss sebagai safety net overfitting")
    print("=" * 55)

    set_trainable = False
    for layer in base_model.layers:
        if layer.name == 'block4_conv1':
            set_trainable = True
        layer.trainable = set_trainable

    model.compile(optimizer=SGD(learning_rate=1e-6, momentum=0.9),
                  loss=loss_fn, metrics=metrik)

    history_p3 = model.fit(
        train_gen, validation_data=val_gen, epochs=15,
        callbacks=[
            EarlyStopping(monitor='val_loss', mode='min', patience=3,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', mode='min',
                              factor=0.5, patience=2, min_lr=1e-8, verbose=1),
            ckpt
        ]
    )

    print("\nTraining 3 fase VGG19 + Channel Attention selesai.")

elif ARCHITECTURE_MODE == 'reference_cnn':
    print("=" * 55)
    print("  TRAINING REFERENCE-LIKE CUSTOM CNN")
    print("  Metodologi lain tetap sama: CLAHE, tanpa crop, augmentasi, bottleneck 200 fitur")
    print("=" * 55)

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=loss_fn,
        metrics=metrik
    )

    history_p1 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=80,
        callbacks=[
            EarlyStopping(monitor='val_auc', mode='max', patience=15,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', mode='min',
                              factor=0.5, patience=6, min_lr=1e-7, verbose=1),
            ckpt,
            csv_logger
        ],
        verbose=1
    )

    print("\nTraining Reference-like Custom CNN selesai.")

if os.path.exists(BEST_MODEL_PATH):
    model.load_weights(BEST_MODEL_PATH)
    print(f"Bobot terbaik dimuat: {BEST_MODEL_PATH}")
else:
    print("Checkpoint belum ditemukan. Model memakai bobot terakhir dari proses training.")

## 7 — Visualisasi Kurva Training

In [ ]:
def gabung(*hs):
    out = {}
    for h in hs:
        if h is None:
            continue
        for k, v in h.history.items():
            out.setdefault(k, []).extend(v)
    return out

histories = [h for h in [history_p1, history_p2, history_p3] if h is not None]
hist_all = gabung(*histories)

print("Metric yang tersedia:")
print(list(hist_all.keys()))

if 'accuracy' not in hist_all:
    raise KeyError("History tidak memiliki key 'accuracy'. Pastikan training sudah berjalan.")

epochs_total = range(1, len(hist_all['accuracy']) + 1)

phase_lengths = [len(h.history['accuracy']) for h in histories]
phase_boundaries = np.cumsum(phase_lengths)[:-1]

plot_candidates = [
    ('accuracy', 'Accuracy', None),
    ('auc', 'AUC Val Target >= 0.85', 0.85),
    ('loss', 'Loss', None),
    ('precision', 'Precision', None)
]

plot_metrics = []
for m, label, target in plot_candidates:
    if m in hist_all and f'val_{m}' in hist_all:
        plot_metrics.append((m, label, target))
    else:
        print(f"Metric '{m}' tidak ditemukan, dilewati.")

if len(plot_metrics) < 4 and 'recall' in hist_all and 'val_recall' in hist_all:
    plot_metrics.append(('recall', 'Recall', None))

plot_metrics = plot_metrics[:4]

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flat
fig.suptitle(
    f'Kurva Training {ARCH_TAG}\n(CLAHE | Tanpa Crop | Label Smoothing 0.05)',
    fontsize=13,
    fontweight='bold'
)

for ax, (m, label, target) in zip(axes, plot_metrics):
    ax.plot(epochs_total, hist_all[m], color='royalblue', lw=2, label='Train')
    ax.plot(epochs_total, hist_all[f'val_{m}'], color='orange', lw=2, label='Val')

    for i, boundary in enumerate(phase_boundaries, start=1):
        ax.axvline(
            x=boundary,
            linestyle='--',
            alpha=0.6,
            label=f'Akhir Fase {i}'
        )

    if target is not None:
        ax.axhline(y=target, color='red', linestyle=':', alpha=0.5, label='Target')

    t = hist_all[m][-1]
    v = hist_all[f'val_{m}'][-1]
    ax.text(
        0.98, 0.05,
        f'Gap: {abs(t-v):.3f}',
        transform=ax.transAxes,
        ha='right',
        fontsize=9,
        color='red',
        fontweight='bold'
    )

    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

for ax in list(axes)[len(plot_metrics):]:
    ax.axis('off')

plt.tight_layout()
plot_path = SAVE_MODEL + f'/kurva_training_{ARCH_TAG}.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Kurva training disimpan: {plot_path}")

if 'val_auc' in hist_all:
    best_val_auc = max(hist_all['val_auc'])
    print(f"\nVal AUC terbaik : {best_val_auc:.4f}")
    print(f"Gap AUC akhir   : {abs(hist_all['auc'][-1] - hist_all['val_auc'][-1]):.4f}")

    if best_val_auc >= 0.85:
        print("Target Val AUC >= 0.85 tercapai. Lanjut ke ekstraksi fitur.")
    else:
        print("Val AUC < 0.85. Pertimbangkan learning rate, jumlah epoch, atau regularisasi.")

## 8 — Ekstraksi Fitur 200 Dimensi (Bottleneck Linear)

In [ ]:
if os.path.exists(BEST_MODEL_PATH):
    model.load_weights(BEST_MODEL_PATH)
    print(f"Bobot terbaik dimuat: {BEST_MODEL_PATH}")
else:
    print("Checkpoint terbaik belum ditemukan. Ekstraksi memakai bobot model saat ini.")

extractor = Model(
    inputs=model.input,
    outputs=model.get_layer("bottleneck_features").output,
    name=f"{ARCH_TAG}_feature_extractor_200F"
)

print(f"=== EKSTRAKSI FITUR 200 DIMENSI: {ARCH_TAG} ===")

FEATURE_NAMES = [f"F{i:03d}" for i in range(N_FEATURES)]

ext_configs = [
    (X_train_bal,  y_train_bal,  aug_train, "Train"),
    (X_val_paths,  y_val_labels, None,      "Val"),
    (X_test_paths, y_test_labels, None,      "Test"),
]

feats_all = {}

def save_feature_report(features, labels, paths, split_name):
    features = features[:len(labels)].astype("float32")

    df_feat = pd.DataFrame(features, columns=FEATURE_NAMES)
    df_feat.insert(0, "Grade", labels)
    df_feat.insert(1, "Nama Grade", [GRADE_SHORT[g] for g in labels])
    df_feat.insert(2, "Nama File", [os.path.basename(p) for p in paths])
    df_feat.insert(3, "Path", paths)

    filename = f"04_fitur_{ARCH_TAG}_{split_name.lower()}_200_dimensi.csv"
    save_and_show(
        df_feat,
        filename,
        n=10,
        title=f"LAPORAN 04: Preview Fitur Ekstraksi {split_name} ({ARCH_TAG})"
    )

    df_stat = pd.DataFrame({
        "Fitur": FEATURE_NAMES,
        "Mean": features.mean(axis=0),
        "Std": features.std(axis=0),
        "Min": features.min(axis=0),
        "Max": features.max(axis=0),
        "Skewness": pd.DataFrame(features, columns=FEATURE_NAMES).skew().values
    })

    stat_filename = f"05_statistik_fitur_{ARCH_TAG}_{split_name.lower()}.csv"
    save_and_show(
        df_stat,
        stat_filename,
        n=10,
        title=f"LAPORAN 05: Statistik Fitur {split_name} ({ARCH_TAG})"
    )

    plt.figure(figsize=(16, 7))
    sns.heatmap(
        pd.DataFrame(features[:30, :50], columns=FEATURE_NAMES[:50]),
        cmap="viridis",
        cbar=True
    )
    plt.title(f"Heatmap 30 Sampel Pertama dan 50 Fitur Pertama - {split_name} ({ARCH_TAG})")
    plt.xlabel("Fitur")
    plt.ylabel("Sampel")
    plt.tight_layout()

    heatmap_path = os.path.join(REPORT_DIR, f"06_heatmap_fitur_{ARCH_TAG}_{split_name.lower()}.png")
    plt.savefig(heatmap_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Heatmap fitur disimpan: {heatmap_path}")

    return features

for X_paths, y_lbls, aug_flags, name in ext_configs:
    print(f"\nMengekstraksi fitur: {name}")

    gen = DataGenerator(
        X_paths,
        y_lbls,
        augment_flags=aug_flags,
        batch_size=BATCH_SIZE,
        shuffle=False,
        save_dir=None
    )

    feats = extractor.predict(gen, verbose=1)
    feats = feats[:len(y_lbls)].astype("float32")

    feats_all[name] = (feats, y_lbls)

    print(f"{name}: {feats.shape}")

    save_feature_report(
        features=feats,
        labels=y_lbls,
        paths=X_paths,
        split_name=name
    )

np.save(SAVE_MODEL + "/X_train_200.npy", feats_all["Train"][0])
np.save(SAVE_MODEL + "/y_train.npy",     feats_all["Train"][1])
np.save(SAVE_MODEL + "/X_val_200.npy",   feats_all["Val"][0])
np.save(SAVE_MODEL + "/y_val.npy",       feats_all["Val"][1])
np.save(SAVE_MODEL + "/X_test_200.npy",  feats_all["Test"][0])
np.save(SAVE_MODEL + "/y_test.npy",      feats_all["Test"][1])

np.save(SAVE_MODEL + f"/X_train_200_{ARCH_TAG}.npy", feats_all["Train"][0])
np.save(SAVE_MODEL + f"/y_train_{ARCH_TAG}.npy",     feats_all["Train"][1])
np.save(SAVE_MODEL + f"/X_val_200_{ARCH_TAG}.npy",   feats_all["Val"][0])
np.save(SAVE_MODEL + f"/y_val_{ARCH_TAG}.npy",       feats_all["Val"][1])
np.save(SAVE_MODEL + f"/X_test_200_{ARCH_TAG}.npy",  feats_all["Test"][0])
np.save(SAVE_MODEL + f"/y_test_{ARCH_TAG}.npy",      feats_all["Test"][1])

extractor.save(SAVE_MODEL + f"/{ARCH_TAG}_extractor_200F.h5")

print("\nSemua fitur 200 dimensi berhasil diekstraksi dan disimpan.")

## 9 — Standardisasi & Validasi Distribusi Fitur

In [ ]:
X_train = np.load(SAVE_MODEL+"/X_train_200.npy")
y_train = np.load(SAVE_MODEL+"/y_train.npy")
X_val   = np.load(SAVE_MODEL+"/X_val_200.npy")
y_val   = np.load(SAVE_MODEL+"/y_val.npy")
X_test  = np.load(SAVE_MODEL+"/X_test_200.npy")
y_test  = np.load(SAVE_MODEL+"/y_test.npy")

X_train=X_train[:len(y_train)]; X_val=X_val[:len(y_val)]; X_test=X_test[:len(y_test)]

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
joblib.dump(scaler, SAVE_MODEL+"/scaler.pkl")
joblib.dump(scaler, SAVE_MODEL+f"/scaler_{ARCH_TAG}.pkl")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribusi Fitur Sebelum & Sesudah Scaling\n'
              '(Bottleneck Linear → Harus Lebih Simetris dari ReLU)',
              fontsize=13, fontweight='bold')

for i, feat_idx in enumerate([0, 50, 100]):
    axes[0, i].hist(X_train[:, feat_idx], bins=60, color='orange',
                     alpha=0.8, edgecolor='black')
    axes[0, i].set_title(f'Fitur {feat_idx} — Sebelum Scaling')
    skew_b = pd.Series(X_train[:,feat_idx]).skew()
    axes[0, i].text(0.98,0.95,f'Skew={skew_b:.2f}',
                     transform=axes[0,i].transAxes,ha='right',fontsize=9,color='red')

    axes[1, i].hist(X_train_scaled[:, feat_idx], bins=60, color='steelblue',
                     alpha=0.8, edgecolor='black')
    axes[1, i].set_title(f'Fitur {feat_idx} — Sesudah Scaling')
    mu=X_train_scaled[:,feat_idx].mean(); sd=X_train_scaled[:,feat_idx].std()
    skew_a=pd.Series(X_train_scaled[:,feat_idx]).skew()
    axes[1, i].text(0.98,0.95,f'μ={mu:.2f} σ={sd:.2f}\nSkew={skew_a:.2f}',
                     transform=axes[1,i].transAxes,ha='right',fontsize=9,color='darkblue')

plt.tight_layout()
plt.savefig(SAVE_MODEL+f'/distribusi_fitur_{ARCH_TAG}.png', dpi=200, bbox_inches='tight')
plt.show()

avg_skew = np.mean([abs(pd.Series(X_train_scaled[:,i]).skew()) for i in range(0,200,10)])
print(f"Rata-rata |skewness| fitur: {avg_skew:.3f}")
print("Target: < 1.0 (simetris cukup untuk SVM)")
print("Fitur siap untuk GA." if avg_skew < 1.5 else "⚠️ Skewness masih tinggi")

df_scaling_summary = pd.DataFrame({
    "Fitur": FEATURE_NAMES,
    "Mean Sebelum Scaling": X_train.mean(axis=0),
    "Std Sebelum Scaling": X_train.std(axis=0),
    "Mean Sesudah Scaling": X_train_scaled.mean(axis=0),
    "Std Sesudah Scaling": X_train_scaled.std(axis=0)
})

save_and_show(
    df_scaling_summary,
    f"07_ringkasan_standardisasi_fitur_{ARCH_TAG}.csv",
    n=15,
    title="LAPORAN 06: Ringkasan Standardisasi Fitur"
)

df_train_scaled_preview = pd.DataFrame(X_train_scaled, columns=FEATURE_NAMES)
df_train_scaled_preview.insert(0, "Grade", y_train)
df_train_scaled_preview.insert(1, "Nama Grade", [GRADE_SHORT[g] for g in y_train])

save_and_show(
    df_train_scaled_preview,
    f"08_fitur_train_scaled_200_dimensi_{ARCH_TAG}.csv",
    n=10,
    title="LAPORAN 07: Preview Fitur Train Setelah Standardisasi"
)

## 10 — Selection Feature using Genetic Algorithm

In [ ]:
class FitnessEvaluator:
    def __init__(self, X, y, seed=SEED, monitor=True, log_every=20, cv_verbose=0):
        self.X = X
        self.y = y
        self.clf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=1)
        self.cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        self.cache = {}
        self.n_calls = 0
        self.n_cached = 0
        self.monitor = monitor
        self.log_every = log_every
        self.cv_verbose = cv_verbose
        self.t0 = time.time()

    def evaluate(self, chromosome):
        key = chromosome.tobytes()

        if key in self.cache:
            self.n_cached += 1
            return self.cache[key]

        self.n_calls += 1
        n_active = int(np.sum(chromosome))

        if n_active == 0:
            self.cache[key] = 0.0
            return 0.0

        if self.monitor and (self.n_calls == 1 or self.n_calls % self.log_every == 0):
            elapsed = (time.time() - self.t0) / 60
            print(
                f"[GA Fitness] evaluasi nyata ke-{self.n_calls} | "
                f"fitur aktif={n_active}/{self.X.shape[1]} | "
                f"cache={self.n_cached} | elapsed={elapsed:.2f} menit",
                flush=True
            )

        s = cross_val_score(
            self.clf,
            self.X[:, chromosome == 1],
            self.y,
            cv=self.cv,
            scoring='roc_auc_ovr',
            n_jobs=-1,
            verbose=self.cv_verbose
        ).mean()

        self.cache[key] = s
        return s

    def stats(self):
        t = self.n_calls + self.n_cached
        print(
            f"  Evaluasi nyata:{self.n_calls} | "
            f"Cache:{self.n_cached} ({self.n_cached / max(t, 1) * 100:.0f}%)"
        )

def init_population(pop_size, n_features, min_active=0.1, seed=SEED):
    rng = np.random.RandomState(seed)
    pop = np.zeros((pop_size, n_features), dtype=int)
    for i in range(pop_size):
        n_act = max(1, int(rng.uniform(min_active, 0.5) * n_features))
        pop[i, rng.choice(n_features, size=n_act, replace=False)] = 1
    return pop

def tournament_selection(pop, scores, ts=5, seed=None):
    rng = np.random.RandomState(seed)
    cands = rng.choice(len(pop), size=ts, replace=False)
    return pop[cands[np.argmax(scores[cands])]].copy()

def uniform_crossover(p1, p2, cp=0.8, cip=0.5, seed=None):
    rng = np.random.RandomState(seed)
    child = p1.copy()
    if rng.rand() < cp:
        mask = rng.rand(len(p1)) < cip
        child[mask] = p2[mask]
    return child

def bit_flip_mutation(c, mp=0.1, mip=0.05, seed=None):
    rng = np.random.RandomState(seed)
    m = c.copy()
    if rng.rand() < mp:
        mask = rng.rand(len(m)) < mip
        m[mask] = 1 - m[mask]
        if np.sum(m) == 0:
            m = c.copy()
    return m

def apply_elitism(op, os_, np_, ns_, ne=1):
    ei = np.argsort(os_)[::-1][:ne]
    wi = np.argsort(ns_)[:ne]
    for i, e in zip(wi, ei):
        np_[i] = op[e].copy()
        ns_[i] = os_[e]
    return np_, ns_

def run_ga(
    evaluator, n_features, pop_size=100, max_gen=50,
    n_no_change=10, ts=5, cp=0.8, cip=0.5, mp=0.1, mip=0.05,
    n_elite=1, seed=SEED, verbose=True, progress_every=10
):
    rng = np.random.RandomState(seed)
    t0 = time.time()

    pop = init_population(pop_size, n_features, seed=seed)

    if verbose:
        print(f"[GA] Evaluasi populasi awal: {pop_size} kromosom", flush=True)

    fit_list = []
    for i, c in enumerate(pop, start=1):
        fit_list.append(evaluator.evaluate(c))
        if verbose and (i % progress_every == 0 or i == pop_size):
            print(f"[GA] Populasi awal {i}/{pop_size} selesai", flush=True)

    fit = np.array(fit_list)

    bi = np.argmax(fit)
    bc = pop[bi].copy()
    bs = fit[bi]

    hist = {
        "best_score": [bs],
        "mean_score": [np.mean(fit)],
        "std_score": [np.std(fit)],
        "n_features": [int(np.sum(bc))],
        "no_change": 0,
        "stopped_gen": max_gen
    }

    if verbose:
        print(f"  Gen  0 | AUC:{bs:.4f} | F:{int(np.sum(bc))}")

    for gen in range(1, max_gen + 1):
        gen_t0 = time.time()
        np_ = np.zeros_like(pop)

        if verbose:
            print(f"\n[GA] Mulai generasi {gen}/{max_gen}", flush=True)

        for i in range(pop_size):
            p1 = tournament_selection(pop, fit, ts, seed=rng.randint(0, 9999))
            p2 = tournament_selection(pop, fit, ts, seed=rng.randint(0, 9999))
            ch = uniform_crossover(p1, p2, cp, cip, seed=rng.randint(0, 9999))
            ch = bit_flip_mutation(ch, mp, mip, seed=rng.randint(0, 9999))
            np_[i] = ch

        nf_list = []
        for i, c in enumerate(np_, start=1):
            nf_list.append(evaluator.evaluate(c))
            if verbose and (i % progress_every == 0 or i == pop_size):
                elapsed = (time.time() - t0) / 60
                print(
                    f"[GA] Gen {gen}/{max_gen} | kromosom {i}/{pop_size} | "
                    f"elapsed={elapsed:.2f} menit",
                    flush=True
                )

        nf = np.array(nf_list)
        np_, nf = apply_elitism(pop, fit, np_, nf, n_elite)

        pop = np_
        fit = nf
        gs = fit[np.argmax(fit)]

        if gs > bs:
            bs = gs
            bc = pop[np.argmax(fit)].copy()
            hist["no_change"] = 0
        else:
            hist["no_change"] += 1

        hist["best_score"].append(bs)
        hist["mean_score"].append(np.mean(fit))
        hist["std_score"].append(np.std(fit))
        hist["n_features"].append(int(np.sum(bc)))

        if verbose:
            gen_elapsed = (time.time() - gen_t0) / 60
            total_elapsed = (time.time() - t0) / 60
            eta = (total_elapsed / max(gen, 1)) * (max_gen - gen)
            print(
                f"  Gen {gen:2d} | AUC:{bs:.4f} | F:{int(np.sum(bc))} | "
                f"NC:{hist['no_change']}/{n_no_change} | "
                f"GenTime:{gen_elapsed:.2f} menit | Elapsed:{total_elapsed:.2f} menit | "
                f"ETA≈{eta:.2f} menit",
                flush=True
            )

        if hist["no_change"] >= n_no_change:
            hist["stopped_gen"] = gen
            if verbose:
                print(f"  ⏹ Early stop gen {gen}")
            break

    if verbose:
        print(
            f"\n GA {(time.time() - t0) / 60:.1f}mnt | "
            f"AUC={bs:.4f} | {int(np.sum(bc))}/{n_features}F "
            f"({(1 - np.sum(bc) / n_features) * 100:.0f}% reduksi)"
        )
        evaluator.stats()

    return bc, bs, hist

print("Komponen GA siap dengan monitoring.")

## Cell 11 — Jalankan GA Seleksi Fitur

In [ ]:
print("=== GA SELEKSI FITUR ===")

SELECTED_FEATURES_PATH = SAVE_MODEL + f"/selected_features_ga_{ARCH_TAG}.npy"

if (not FORCE_RERUN_GA) and os.path.exists(SELECTED_FEATURES_PATH):
    idx_ga = np.load(SELECTED_FEATURES_PATH)
    ga_chromosome = np.zeros(N_FEATURES, dtype=int)
    ga_chromosome[idx_ga] = 1
    ga_score = np.nan
    ga_history = None

    print(f"Fitur GA ditemukan dan dimuat ulang: {SELECTED_FEATURES_PATH}")
    print(f"GA → {len(idx_ga)} fitur dari {N_FEATURES} ({(1-len(idx_ga)/N_FEATURES)*100:.0f}% reduksi)")
    print("Catatan: GA tidak dijalankan ulang. Ubah FORCE_RERUN_GA=True jika ingin proses GA ulang.")
else:
    evaluator_ga = FitnessEvaluator(
        X_train_scaled,
        y_train,
        seed=SEED,
        monitor=True,
        log_every=20,
        cv_verbose=0
    )

    ga_chromosome, ga_score, ga_history = run_ga(
        evaluator_ga,
        N_FEATURES,
        pop_size=100,
        max_gen=50,
        n_no_change=10,
        seed=SEED,
        verbose=True,
        progress_every=10
    )

    idx_ga = np.where(ga_chromosome == 1)[0]

    np.save(SELECTED_FEATURES_PATH, idx_ga)
    np.save(SAVE_MODEL + "/selected_features_ga.npy", idx_ga)

X_train_ga = X_train_scaled[:, idx_ga]
X_val_ga   = X_val_scaled[:, idx_ga]
X_test_ga  = X_test_scaled[:, idx_ga]

print(f"\nGA → {len(idx_ga)} fitur dari {N_FEATURES} ({(1-len(idx_ga)/N_FEATURES)*100:.0f}% reduksi)")
if not np.isnan(ga_score):
    print(f"Fitness AUC: {ga_score:.4f}")
else:
    print("Fitness AUC: dimuat dari file, skor GA tidak dihitung ulang.")
print("Shape X_train_ga:", X_train_ga.shape)
print("Shape X_val_ga  :", X_val_ga.shape)
print("Shape X_test_ga :", X_test_ga.shape)

if ga_history is not None:
    df_ga_history = pd.DataFrame({
        "Generasi": range(len(ga_history["best_score"])),
        "Best AUC": ga_history["best_score"],
        "Mean AUC": ga_history["mean_score"],
        "Std AUC": ga_history["std_score"],
        "Jumlah Fitur Aktif": ga_history["n_features"]
    })

    save_and_show(
        df_ga_history,
        f"09_riwayat_evolusi_ga_{ARCH_TAG}.csv",
        n=20,
        title="LAPORAN 08: Riwayat Evolusi Genetic Algorithm"
    )
else:
    print("Riwayat evolusi GA tidak dibuat karena fitur GA dimuat dari file.")

df_selected_features = pd.DataFrame({
    "Urutan": range(1, len(idx_ga) + 1),
    "Index Fitur": idx_ga,
    "Nama Fitur": [f"F{i:03d}" for i in idx_ga]
})

save_and_show(
    df_selected_features,
    f"10_fitur_terseleksi_ga_{ARCH_TAG}.csv",
    n=50,
    title="LAPORAN 09: Daftar Fitur yang Terseleksi oleh GA"
)

df_feature_mask = pd.DataFrame({
    "Index Fitur": range(N_FEATURES),
    "Nama Fitur": FEATURE_NAMES,
    "Dipilih GA": ["Ya" if i in set(idx_ga) else "Tidak" for i in range(N_FEATURES)]
})

save_and_show(
    df_feature_mask,
    f"11_mask_fitur_ga_{ARCH_TAG}.csv",
    n=30,
    title="LAPORAN 10: Mask Seleksi Semua Fitur"
)

if ga_history is not None:
    plt.figure(figsize=(12, 6))
    plt.plot(df_ga_history["Generasi"], df_ga_history["Best AUC"], marker="o", label="Best AUC")
    plt.plot(df_ga_history["Generasi"], df_ga_history["Mean AUC"], marker="s", label="Mean AUC")
    plt.fill_between(
        df_ga_history["Generasi"],
        df_ga_history["Mean AUC"] - df_ga_history["Std AUC"],
        df_ga_history["Mean AUC"] + df_ga_history["Std AUC"],
        alpha=0.2,
        label="± Std"
    )
    plt.title("Evolusi Fitness Genetic Algorithm")
    plt.xlabel("Generasi")
    plt.ylabel("ROC-AUC OVR")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    ga_plot_path = os.path.join(REPORT_DIR, f"12_evolusi_ga_{ARCH_TAG}.png")
    plt.savefig(ga_plot_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Grafik evolusi GA disimpan: {ga_plot_path}")

print("Seleksi fitur GA siap digunakan untuk tuning model.")

## 12 — Hyperparameter Tuning SVM & RF

In [ ]:
CLASSIFIER_TUNING_EXPERIMENT = "raza_2024_inspired_randomized_search"

SVM_TUNING_N_ITER = 24
RF_TUNING_N_ITER  = 32
CV_FOLDS_TUNING   = 5

def summarize_search_results(search_obj, filename, title):
    """Simpan ringkasan hasil RandomizedSearchCV agar proses tuning bisa diaudit."""
    df_cv = pd.DataFrame(search_obj.cv_results_).sort_values("rank_test_score")

    cols = [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "std_train_score",
        "mean_fit_time",
        "std_fit_time",
        "params"
    ]
    cols = [c for c in cols if c in df_cv.columns]

    save_and_show(
        df_cv[cols],
        filename,
        n=min(20, len(df_cv)),
        title=title
    )

    return df_cv

CV_TUNING = StratifiedKFold(
    n_splits=CV_FOLDS_TUNING,
    shuffle=True,
    random_state=SEED
)

PARAM_SVM = {
    "C": [0.1, 1, 10, 50, 70, 100],
    "gamma": ["scale", "auto", 0.001, 0.01],
    "kernel": ["linear", "poly", "rbf", "sigmoid"],
    "class_weight": [None, "balanced"],
}

PARAM_RF = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 4, 8, 12],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "bootstrap": [True],
    "class_weight": [None, "balanced", "balanced_subsample"],
    "warm_start": [False, True],
    "n_jobs": [1],
}

print("Eksperimen tuning classifier:", CLASSIFIER_TUNING_EXPERIMENT)
print("SVM_TUNING_N_ITER:", SVM_TUNING_N_ITER)
print("RF_TUNING_N_ITER:", RF_TUNING_N_ITER)
print("CV_FOLDS_TUNING:", CV_FOLDS_TUNING)

def tune_svm(
    Xtr, ytr, tag="",
    n_iter=SVM_TUNING_N_ITER, cv=CV_TUNING,
    verbose=MONITOR_VERBOSE, n_jobs=N_JOBS_TUNING
):
    total_fits = n_iter * CV_FOLDS_TUNING
    log_step(
        f"Mulai Tuning SVM ({tag}) | kandidat={n_iter} | CV={CV_FOLDS_TUNING} | "
        f"total fit={total_fits} | n_jobs={n_jobs} | verbose={verbose} | "
        f"scoring=roc_auc_ovr | probability=True"
    )

    t0 = time.time()

    search_estimator = SVC(
        probability=True,
        random_state=SEED,
        cache_size=2000
    )

    search = RandomizedSearchCV(
        estimator=search_estimator,
        param_distributions=PARAM_SVM,
        n_iter=n_iter,
        cv=cv,
        scoring='roc_auc_ovr',
        n_jobs=n_jobs,
        verbose=verbose,
        random_state=SEED,
        return_train_score=True,
        error_score='raise',
        pre_dispatch='2*n_jobs'
    )

    search.fit(Xtr, ytr)

    elapsed = time.time() - t0
    log_step(
        f"Selesai Tuning SVM ({tag}) | durasi={fmt_minutes(elapsed)} | "
        f"Best={search.best_params_} | CV-AUC={search.best_score_:.4f}"
    )

    summarize_search_results(
        search,
        f"12_cv_results_svm_{tag.lower()}_{ARCH_TAG}.csv",
        f"LAPORAN: Hasil Tuning SVM ({tag}) ({ARCH_TAG})"
    )

    return search.best_estimator_

def tune_rf(
    Xtr, ytr, tag="",
    n_iter=RF_TUNING_N_ITER, cv=CV_TUNING,
    verbose=MONITOR_VERBOSE, n_jobs=N_JOBS_TUNING
):
    total_fits = n_iter * CV_FOLDS_TUNING
    log_step(
        f"Mulai Tuning RF ({tag}) | kandidat={n_iter} | CV={CV_FOLDS_TUNING} | "
        f"total fit={total_fits} | n_jobs={n_jobs} | verbose={verbose} | scoring=roc_auc_ovr"
    )

    t0 = time.time()

    search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=SEED),
        param_distributions=PARAM_RF,
        n_iter=n_iter,
        cv=cv,
        scoring='roc_auc_ovr',
        n_jobs=n_jobs,
        verbose=verbose,
        random_state=SEED,
        return_train_score=True,
        error_score='raise',
        pre_dispatch='2*n_jobs'
    )

    search.fit(Xtr, ytr)

    elapsed = time.time() - t0
    log_step(
        f"Selesai Tuning RF ({tag}) | durasi={fmt_minutes(elapsed)} | "
        f"Best={search.best_params_} | CV-AUC={search.best_score_:.4f}"
    )

    summarize_search_results(
        search,
        f"13_cv_results_rf_{tag.lower()}_{ARCH_TAG}.csv",
        f"LAPORAN: Hasil Tuning Random Forest ({tag}) ({ARCH_TAG})"
    )

    return search.best_estimator_

print(f"=== BASELINE (200 Fitur, Tanpa GA) | {ARCH_TAG} ===")
log_step("Training baseline SVM dimulai.")
svm_base = SVC(
    probability=True,
    random_state=SEED,
    cache_size=2000,
    class_weight="balanced"
)
svm_base.fit(X_train_scaled, y_train)
log_step("Training baseline SVM selesai.")

log_step("Training baseline Random Forest dimulai.")
rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=8,
    max_features="sqrt",
    class_weight="balanced_subsample",
    bootstrap=True,
    random_state=SEED,
    n_jobs=-1
)
rf_base.fit(X_train_scaled, y_train)
log_step("Training baseline Random Forest selesai.")

print("\n=== TUNING DENGAN FITUR GA ===")
best_svm = tune_svm(X_train_ga, y_train, "GA")
best_rf  = tune_rf(X_train_ga, y_train, "GA")

models_to_save = [
    (f"baseline_svm_{ARCH_TAG}.pkl", svm_base),
    (f"baseline_rf_{ARCH_TAG}.pkl", rf_base),
    (f"best_svm_ga_{ARCH_TAG}.pkl", best_svm),
    (f"best_rf_ga_{ARCH_TAG}.pkl", best_rf),
    ("baseline_svm.pkl", svm_base),
    ("baseline_rf.pkl", rf_base),
    ("best_svm_ga.pkl", best_svm),
    ("best_rf_ga.pkl", best_rf),
    ("best_svm_all_features.pkl", svm_base),
    ("best_rf_all_features.pkl", rf_base),
]

for fname, mdl in models_to_save:
    save_path = os.path.join(SAVE_MODEL, fname)
    joblib.dump(mdl, save_path)
    log_step(f"Model disimpan: {save_path}")

print("\nTuning selesai. Semua model disimpan.")

In [ ]:
if hasattr(best_rf, "feature_importances_"):
    df_importance = pd.DataFrame({
        "Index Fitur Asli": idx_ga,
        "Nama Fitur": [f"F{i:03d}" for i in idx_ga],
        "Importance RF": best_rf.feature_importances_
    })

    df_importance = df_importance.sort_values(
        by="Importance RF",
        ascending=False
    ).reset_index(drop=True)

    df_importance.insert(0, "Ranking", range(1, len(df_importance) + 1))

    save_and_show(
        df_importance,
        f"14_feature_importance_rf_ga_{ARCH_TAG}.csv",
        n=30,
        title="LAPORAN 12: Feature Importance Random Forest pada Fitur GA"
    )

    plt.figure(figsize=(12, 8))
    top_n = min(30, len(df_importance))

    sns.barplot(
        data=df_importance.head(top_n),
        x="Importance RF",
        y="Nama Fitur"
    )

    plt.title(f"Top {top_n} Feature Importance - Random Forest + GA")
    plt.xlabel("Importance")
    plt.ylabel("Fitur")
    plt.tight_layout()

    importance_path = os.path.join(REPORT_DIR, "15_top_feature_importance_rf_ga.png")
    plt.savefig(importance_path, dpi=200, bbox_inches="tight")
    plt.show()

    print(f"Grafik feature importance disimpan: {importance_path}")
else:
    print("Model RF tidak memiliki atribut feature_importances_.")

## 13 — Evaluasi Lengkap (Fokus Analisis per Grade)

In [ ]:
def evaluate_full(model, name, Xv, yv, Xt, yt, pfx=""):
    for split, Xd, yd in [("Validasi",Xv,yv), ("Test",Xt,yt)]:
        yp=model.predict(Xd); ypr=model.predict_proba(Xd)
        acc=accuracy_score(yd,yp)
        auc_score=roc_auc_score(yd,ypr,multi_class='ovr',average='macro')
        f1w=f1_score(yd,yp,average='weighted')
        sen=np.mean([f1_score(yd==c,yp==c) for c in range(5)])
        print(f"  [{split}] Acc:{acc:.4f} | AUC:{auc_score:.4f} | F1-W:{f1w:.4f}")

    yp_test=model.predict(Xt)
    print(f"\n  Classification Report (Test) — {name}:")
    print(classification_report(yt, yp_test, target_names=GRADE_SHORT, digits=4))

    cm=confusion_matrix(yt,yp_test); cm_n=cm.astype(float)/cm.sum(axis=1,keepdims=True)
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=axes[0],
                xticklabels=GRADE_SHORT,yticklabels=GRADE_SHORT,linewidths=0.5)
    axes[0].set_title(f'Confusion Matrix\n{name}',fontweight='bold')
    axes[0].set_ylabel('Label Sebenarnya'); axes[0].set_xlabel('Label Prediksi')
    sns.heatmap(cm_n,annot=True,fmt='.2f',cmap='Greens',ax=axes[1],
                xticklabels=GRADE_SHORT,yticklabels=GRADE_SHORT,linewidths=0.5,vmin=0,vmax=1)
    axes[1].set_title(f'CM Normalized\n{name}',fontweight='bold')
    axes[1].set_ylabel('Label Sebenarnya'); axes[1].set_xlabel('Label Prediksi')
    plt.tight_layout()
    plt.savefig(SAVE_MODEL+f'/cm_{pfx}.png',dpi=200,bbox_inches='tight'); plt.show()

    yb=label_binarize(yt,classes=[0,1,2,3,4]); ys=model.predict_proba(Xt)
    colors=['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']
    fig,ax=plt.subplots(figsize=(8,7))
    for i,color in enumerate(colors):
        fp,tp,_=roc_curve(yb[:,i],ys[:,i])
        auc_i=auc(fp,tp)
        ax.plot(fp,tp,color=color,lw=2.5,label=f'Grade {i} (AUC={auc_i:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1.5,label='Random Guess')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve (OvR)\n{name}',fontweight='bold')
    ax.legend(loc='lower right',fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(SAVE_MODEL+f'/roc_{pfx}.png',dpi=200,bbox_inches='tight'); plt.show()

    return {
        "acc_val": accuracy_score(yv,model.predict(Xv)),
        "auc_val": roc_auc_score(yv,model.predict_proba(Xv),multi_class='ovr',average='macro'),
        "acc_test": accuracy_score(yt,yp_test),
        "auc_test": roc_auc_score(yt,model.predict_proba(Xt),multi_class='ovr',average='macro'),
    }

print("=== EVALUASI SEMUA MODEL ===")
sep = "="*60
results = {}
for nm, mdl, Xv, Xt, pfx in [
    ("SVM Baseline (200F)", svm_base, X_val_scaled, X_test_scaled, f"svm_base_{ARCH_TAG}"),
    ("RF  Baseline (200F)", rf_base,  X_val_scaled, X_test_scaled, f"rf_base_{ARCH_TAG}"),
    ("SVM + GA",            best_svm, X_val_ga,     X_test_ga,     f"svm_ga_{ARCH_TAG}"),
    ("RF  + GA",            best_rf,  X_val_ga,     X_test_ga,     f"rf_ga_{ARCH_TAG}"),
]:
    print(f"\n{sep}\n  {nm}\n{sep}")
    results[nm] = evaluate_full(mdl, nm, Xv, y_val, Xt, y_test, pfx)

In [ ]:
def save_prediction_detail(model, model_name, X_data, y_true, image_paths, filename):
    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)

    df_pred = pd.DataFrame({
        "Nama File": [os.path.basename(p) for p in image_paths],
        "Path": image_paths,
        "Grade Aktual": y_true,
        "Nama Grade Aktual": [GRADE_SHORT[g] for g in y_true],
        "Grade Prediksi": y_pred,
        "Nama Grade Prediksi": [GRADE_SHORT[g] for g in y_pred],
        "Benar/Salah": ["Benar" if a == b else "Salah" for a, b in zip(y_true, y_pred)],
        "Confidence Maksimum": np.max(y_prob, axis=1)
    })

    for g in range(5):
        df_pred[f"Probabilitas Grade {g}"] = y_prob[:, g]

    save_and_show(
        df_pred,
        filename,
        n=20,
        title=f"LAPORAN 13: Detail Prediksi Test - {model_name}"
    )

    return df_pred

df_pred_rf_ga = save_prediction_detail(
    model=best_rf,
    model_name="Random Forest + GA",
    X_data=X_test_ga,
    y_true=y_test,
    image_paths=X_test_paths,
    filename="16_detail_prediksi_test_rf_ga.csv"
)

df_pred_svm_ga = save_prediction_detail(
    model=best_svm,
    model_name="SVM + GA",
    X_data=X_test_ga,
    y_true=y_test,
    image_paths=X_test_paths,
    filename="17_detail_prediksi_test_svm_ga.csv"
)

## 14 — Tabel Ringkasan & Perbandingan dengan Paper Sam Chandra Bose

In [ ]:
fitur_map = {
    "SVM Baseline (200F)": N_FEATURES,
    "RF  Baseline (200F)": N_FEATURES,
    "SVM + GA": len(idx_ga),
    "RF  + GA": len(idx_ga)
}

rows = []
for nm, r in results.items():
    n_fitur = fitur_map.get(nm, N_FEATURES)
    rows.append({
        "Arsitektur": ARCH_TAG,
        "Model": nm,
        "Fitur": n_fitur,
        "Reduksi": f"{(N_FEATURES-n_fitur)/N_FEATURES*100:.0f}%",
        "AUC Val": f"{r['auc_val']:.4f}",
        "AUC Test": f"{r['auc_test']:.4f}",
        "Acc Test": f"{r['acc_test']*100:.2f}%",
    })

df = pd.DataFrame(rows)
print("\n" + "="*75)
print("  RINGKASAN HASIL PENELITIAN")
print("="*75)
print(df.to_string(index=False))

print("\n" + "="*75)
print("  PERBANDINGAN DENGAN PAPER ACUAN SAM CHANDRA BOSE")
print("="*75)
print(f"  {'Model':<32} {'Accuracy':>12}  Catatan")
print(f"  {'-'*70}")
print(f"  {'Paper acuan RF multi-class':<32} {'96.38%':>12}  CNN custom 200 fitur + RF")
print()
for nm, r in results.items():
    print(f"  {nm:<32} {r['acc_test']*100:>11.2f}%  Dataset dan metodologi penelitian Anda")

summary_path = SAVE_MODEL + f"/ringkasan_hasil_{ARCH_TAG}.csv"
df.to_csv(summary_path, index=False)
df.to_csv(SAVE_MODEL + "/ringkasan_hasil.csv", index=False)
print(f"\nRingkasan disimpan: {summary_path}")

## 15 — Simpan Semua Model & Artefak Final

In [ ]:
print("=== MENYIMPAN SEMUA ARTEFAK ===")

files_expected = [
    f"{ARCH_TAG}_best.weights.h5",
    f"{ARCH_TAG}_extractor_200F.h5",
    f"scaler_{ARCH_TAG}.pkl",
    f"selected_features_ga_{ARCH_TAG}.npy",
    f"baseline_svm_{ARCH_TAG}.pkl",
    f"baseline_rf_{ARCH_TAG}.pkl",
    f"best_svm_ga_{ARCH_TAG}.pkl",
    f"best_rf_ga_{ARCH_TAG}.pkl",
    f"ringkasan_hasil_{ARCH_TAG}.csv",
    f"kurva_training_{ARCH_TAG}.png",
    f"distribusi_fitur_{ARCH_TAG}.png",
]

print(f"\nFile utama di {SAVE_MODEL}:")
for fname in files_expected:
    path = SAVE_MODEL + "/" + fname
    status = "Ada" if os.path.exists(path) else "Belum ada"
    print(f"  {status}: {fname}")

print("\nPipeline selesai.")
print()
print("=== RANGKUMAN PIPELINE ===")
print(f"  1. Mode arsitektur : {ARCH_TAG}")
print("  2. Preprocessing   : CLAHE clip=2.0, tile=8x8, bicubic 224x224, tanpa crop")
print("  3. Balancing       : RandomUnderSampler + augmentasi sampai 1000/kelas")
if ARCHITECTURE_MODE == 'vgg19_attention':
    print("  4. Arsitektur      : VGG19 + Channel Attention + Bottleneck 200 fitur")
    print("  5. Training        : 3 fase, warm-up, block 5, block 4+5")
else:
    print("  4. Arsitektur      : Custom CNN pembanding, 5 conv, 3 pooling, bottleneck 200 fitur")
    print("  5. Training        : Single phase custom CNN")
print("  6. Loss            : CategoricalCrossentropy label_smoothing=0.05")
print(f"  7. GA              : ROC-AUC OvR fitness, fitur terpilih {len(idx_ga)}/{N_FEATURES}")
print("  8. Klasifikasi     : SVM probability=True dan Random Forest")
print("  9. Validasi        : Validation set, test set, dan data primer RSUA bila tersedia")